In [1]:
# -*- coding: utf-8 -*-
"""
中文NER任务主程序
"""
import os
import sys
import torch
import numpy as np
from transformers import BertTokenizer, BertForTokenClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset
from typing import List, Dict, Tuple

def check_and_download_model():
    """检查并下载模型"""
    LOCAL_MODEL_PATH = "./bert-base-chinese-local"
    required_files = ['config.json', 'pytorch_model.bin', 'tokenizer_config.json', 'vocab.txt']
    
    print("=" * 60)
    print("检查本地模型文件...")
    
    # 检查所有必需文件是否存在
    all_files_exist = all(
        os.path.exists(os.path.join(LOCAL_MODEL_PATH, f))
        for f in required_files
    )
    
    if not all_files_exist:
        print("本地模型文件不完整")
        return False
    
    print("✓ 本地模型文件完整")
    return True

def compute_overall_metrics(predictions: np.ndarray, labels: np.ndarray) -> Dict:
    """
    计算测试集上的整体指标
    Args:
        predictions: 预测的标签ID数组，形状为 (batch_size, seq_len)
        labels: 真实的标签ID数组，形状为 (batch_size, seq_len)
    """
    # 将预测结果展平为一维数组
    pred_flat = predictions.flatten()
    label_flat = labels.flatten()
    
    # 忽略填充的token（标签为0）
    mask = label_flat != 0
    pred_flat = pred_flat[mask]
    label_flat = label_flat[mask]
    
    # 计算整体指标
    total_tokens = len(label_flat)
    correct_tokens = np.sum(pred_flat == label_flat)
    accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    
    # 计算各个标签的准确率
    unique_labels = np.unique(label_flat)
    label_accuracies = {}
    
    for label_id in unique_labels:
        if label_id == 0:  # 忽略填充标签
            continue
        mask = label_flat == label_id
        label_total = np.sum(mask)
        label_correct = np.sum(pred_flat[mask] == label_flat[mask])
        label_accuracy = label_correct / label_total if label_total > 0 else 0.0
        label_accuracies[label_id] = label_accuracy
    
    return {
        'total_tokens': total_tokens,
        'correct_tokens': correct_tokens,
        'accuracy': accuracy,
        'label_accuracies': label_accuracies
    }

def main():
    """主函数"""
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"\n使用设备: {device}")
    
    # 数据集路径配置
    TRAIN_PATH = "train.char.bmes"
    DEV_PATH = "dev.char.bmes"
    TEST_PATH = "test.char.bmes"
    
    # 模型/训练参数
    MAX_LENGTH = 128
    BATCH_SIZE = 8
    EPOCHS = 10
    LOCAL_MODEL_PATH = "./bert-base-chinese-local"
    OUTPUT_DIR = "./ner_results"
    LOG_DIR = "./ner_logs"
    
    # 创建输出目录
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)
    
    print(f"\n使用本地模型: {LOCAL_MODEL_PATH}")
    print(f"输出目录: {OUTPUT_DIR}")
    print(f"日志目录: {LOG_DIR}")
    
    # ===================== 数据加载函数 =====================
    def load_bmes_data(file_path: str) -> Tuple[List[List[str]], List[List[str]]]:
        """
        加载BMES格式的NER数据集
        输入文件格式：每行是 字符 + 空格 + 标签，空行分隔不同句子
        返回：tokens列表（每个元素是句子的字符列表），labels列表（每个元素是句子的标签列表）
        """
        tokens_list = []
        labels_list = []
        current_tokens = []
        current_labels = []
        
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                # 空行表示句子结束
                if not line:
                    if current_tokens and current_labels:
                        tokens_list.append(current_tokens)
                        labels_list.append(current_labels)
                        current_tokens = []
                        current_labels = []
                    continue
                # 分割字符和标签
                parts = line.split()
                if len(parts) >= 2:
                    char = parts[0]
                    label = parts[1]
                    current_tokens.append(char)
                    current_labels.append(label)
        
        # 处理最后一个句子
        if current_tokens and current_labels:
            tokens_list.append(current_tokens)
            labels_list.append(current_labels)
        
        print(f"加载 {file_path} 完成，共 {len(tokens_list)} 个句子")
        return tokens_list, labels_list
    
    # 加载数据集
    print("\n" + "="*50)
    print("加载数据集...")
    train_tokens, train_labels = load_bmes_data(TRAIN_PATH)
    dev_tokens, dev_labels = load_bmes_data(DEV_PATH)
    test_tokens, test_labels = load_bmes_data(TEST_PATH)
    
    # 检查数据是否为空
    if not train_tokens:
        print("错误: 训练集为空!")
        sys.exit(1)
    
    # ===================== 构建标签映射 =====================
    print("\n" + "="*50)
    print("构建标签映射...")
    all_labels = []
    for labels in train_labels + dev_labels + test_labels:
        all_labels.extend(labels)
    unique_labels = list(set(all_labels))
    label_to_id = {label: idx for idx, label in enumerate(unique_labels)}
    id_to_label = {idx: label for label, idx in label_to_id.items()}
    
    print(f"总标签数: {len(label_to_id)}")
    print("标签示例:")
    for i, (label, idx) in enumerate(list(label_to_id.items())[:10]):
        print(f"  {label} -> {idx}")
    if len(label_to_id) > 10:
        print(f"  ... (共{len(label_to_id)}个标签)")
    
    # ===================== 构建HF数据集 =====================
    print("\n" + "="*50)
    print("构建HF数据集...")
    train_data = {"tokens": train_tokens, "labels": train_labels}
    dev_data = {"tokens": dev_tokens, "labels": dev_labels}
    test_data = {"tokens": test_tokens, "labels": test_labels}
    
    train_dataset = HFDataset.from_dict(train_data)
    dev_dataset = HFDataset.from_dict(dev_data)
    test_dataset = HFDataset.from_dict(test_data)
    
    # ===================== 数据编码 =====================
    print("\n" + "="*50)
    print("加载分词器和编码数据...")
    print(f"从本地加载分词器: {LOCAL_MODEL_PATH}")
    tokenizer = BertTokenizer.from_pretrained(LOCAL_MODEL_PATH)
    
    def encode_dataset(examples: Dict) -> Dict:
        """编码数据集"""
        # 填空1：编码tokens
        # encoding = ______(examples["tokens"], ______=True, ______="max_length", is_split_into_words=True, max_length=MAX_LENGTH)
        encoding = tokenizer(
            examples["tokens"],
            truncation=True,
            padding="max_length",
            is_split_into_words=True,
            max_length=MAX_LENGTH
        )
        
        # 处理标签
        labels = []
        pad_label_id = label_to_id.get("O", 0)
        
        for i, label_seq in enumerate(examples["labels"]):
            # 转换标签为ID
            label_ids = [label_to_id[label] for label in label_seq]
            # 填充到最大长度
            if len(label_ids) < MAX_LENGTH:
                label_ids += [pad_label_id] * (MAX_LENGTH - len(label_ids))
            else:
                label_ids = label_ids[:MAX_LENGTH]
            labels.append(label_ids)
        
        encoding["labels"] = labels
        return encoding
    
    # 编码所有数据集
    print("编码训练集...")
    encoded_train = train_dataset.map(encode_dataset, batched=True)
    print("编码验证集...")
    encoded_dev = dev_dataset.map(encode_dataset, batched=True)
    print("编码测试集...")
    encoded_test = test_dataset.map(encode_dataset, batched=True)
    
    # 设置PyTorch格式
    encoded_train = encoded_train.with_format("torch", device=device)
    encoded_dev = encoded_dev.with_format("torch", device=device)
    encoded_test = encoded_test.with_format("torch", device=device)
    
    print(f"训练集大小: {len(encoded_train)}")
    print(f"验证集大小: {len(encoded_dev)}")
    print(f"测试集大小: {len(encoded_test)}")
    
    # ===================== 加载模型 =====================
    print("\n" + "="*50)
    print(f"从本地加载模型: {LOCAL_MODEL_PATH}")
    # 填空2：加载模型
    # model = BertForTokenClassification.from_pretrained(LOCAL_MODEL_PATH, ______=len(label_to_id), ______=id_to_label, ______=label_to_id, ignore_mismatched_sizes=True)
    model = BertForTokenClassification.from_pretrained(
        LOCAL_MODEL_PATH,
        num_labels=len(label_to_id),
        id2label=id_to_label,
        label2id=label_to_id,
        ignore_mismatched_sizes=True  # 忽略标签数不匹配的警告
    )
    print("✓ 模型加载成功!")

    
    model = model.to(device)
    
    # ===================== 训练参数配置 =====================
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        logging_dir=LOG_DIR,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
        report_to="none",
        learning_rate=2e-5,
        weight_decay=0.01,
    )
    
    # ===================== 初始化Trainer =====================
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_train,
        eval_dataset=encoded_dev,
    )
    
    # ===================== 开始训练 =====================
    print("\n" + "="*50)
    print("开始训练模型...")
    print(f"训练参数:")
    print(f"- Epochs: {EPOCHS}")
    print(f"- Batch Size: {BATCH_SIZE}")
    print(f"- 学习率: 2e-5")
    print(f"- 最大序列长度: {MAX_LENGTH}")
    print(f"- 标签数量: {len(label_to_id)}")
    

    trainer.train()
    print("✓ 训练完成!")

    # ===================== 模型评估 =====================
    print("\n" + "="*50)
    print("评估模型性能...")
    

    eval_results = trainer.evaluate(encoded_test)
    print(f"测试集评估结果:")
    for key, value in eval_results.items():
        print(f"  {key}: {value:.4f}")

    # ===================== 保存最终模型 =====================
    print("\n" + "="*50)
    print("保存训练好的模型...")
    
    final_model_dir = os.path.join(OUTPUT_DIR, "final_model")
    model.save_pretrained(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)
    print(f"✓ 模型已保存到: {final_model_dir}")
    
    # ===================== 在测试集上计算整体指标 =====================
    print("\n" + "="*50)
    print("在测试集上计算整体指标...")
    
    # 获取测试集的预测结果
    predictions = trainer.predict(encoded_test)
    pred_ids = np.argmax(predictions.predictions, axis=2)
    true_ids = predictions.label_ids
    
    # 计算整体指标
    overall_metrics = compute_overall_metrics(pred_ids, true_ids)
    
    print("\n测试集整体指标:")
    print("-" * 50)
    print(f"总token数: {overall_metrics['total_tokens']}")
    print(f"正确预测的token数: {overall_metrics['correct_tokens']}")
    print(f"准确率 (Accuracy): {overall_metrics['accuracy']:.4f}")
    
if __name__ == "__main__":
    main()


使用设备: cuda:0

使用本地模型: ./bert-base-chinese-local
输出目录: ./ner_results
日志目录: ./ner_logs

加载数据集...
加载 train.char.bmes 完成，共 3821 个句子
加载 dev.char.bmes 完成，共 463 个句子
加载 test.char.bmes 完成，共 477 个句子

构建标签映射...
总标签数: 28
标签示例:
  S-ORG -> 0
  M-EDU -> 1
  E-ORG -> 2
  M-TITLE -> 3
  S-RACE -> 4
  B-ORG -> 5
  B-NAME -> 6
  E-PRO -> 7
  M-ORG -> 8
  M-RACE -> 9
  ... (共28个标签)

构建HF数据集...

加载分词器和编码数据...
从本地加载分词器: ./bert-base-chinese-local
编码训练集...
加载 train.char.bmes 完成，共 3821 个句子
加载 dev.char.bmes 完成，共 463 个句子
加载 test.char.bmes 完成，共 477 个句子

构建标签映射...
总标签数: 28
标签示例:
  S-ORG -> 0
  M-EDU -> 1
  E-ORG -> 2
  M-TITLE -> 3
  S-RACE -> 4
  B-ORG -> 5
  B-NAME -> 6
  E-PRO -> 7
  M-ORG -> 8
  M-RACE -> 9
  ... (共28个标签)

构建HF数据集...

加载分词器和编码数据...
从本地加载分词器: ./bert-base-chinese-local
编码训练集...


Map:   0%|          | 0/3821 [00:00<?, ? examples/s]

编码验证集...


Map:   0%|          | 0/463 [00:00<?, ? examples/s]

编码测试集...


...

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./bert-base-chinese-local and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([10, 768]) in the checkpoint and torch.Size([28, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([10]) in the checkpoint and torch.Size([28]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


训练集大小: 3821
验证集大小: 463
测试集大小: 477

从本地加载模型: ./bert-base-chinese-local
✓ 模型加载成功!

开始训练模型...
训练参数:
- Epochs: 10
- Batch Size: 8
- 学习率: 2e-5
- 最大序列长度: 128
- 标签数量: 28

开始训练模型...
训练参数:
- Epochs: 10
- Batch Size: 8
- 学习率: 2e-5
- 最大序列长度: 128
- 标签数量: 28


Epoch,Training Loss,Validation Loss
1,0.024500,0.019664
2,0.028100,0.019101
3,0.009100,0.024068
4,0.003600,0.027956
5,0.000400,0.032652
6,0.002800,0.035430
7,0.000300,0.038024
8,0.000500,0.038783
9,0.002100,0.040086
10,0.002100,0.039844


✓ 训练完成!

评估模型性能...


测试集评估结果:
  eval_loss: 0.0236
  eval_runtime: 0.8876
  eval_samples_per_second: 537.3780
  eval_steps_per_second: 67.5950
  epoch: 10.0000

保存训练好的模型...
✓ 模型已保存到: ./ner_results\final_model

在测试集上计算整体指标...
✓ 模型已保存到: ./ner_results\final_model

在测试集上计算整体指标...

测试集整体指标:
--------------------------------------------------
总token数: 61056
正确预测的token数: 60625
准确率 (Accuracy): 0.9929

测试集整体指标:
--------------------------------------------------
总token数: 61056
正确预测的token数: 60625
准确率 (Accuracy): 0.9929
